# RAG demo — legal context for zero-shot crime classification (AirLLM backend)

Pipeline:
1. Fetch **US Code Title 18** statutes from govinfo.gov (no API key).
2. Fetch real **CourtListener** judgments (two-hop: search → opinions/{id}/, token via `.env`).
3. Fetch **UK Acts of Parliament** from legislation.gov.uk (no key, OGL).
4. Merge all three into one **FAISS** index over statutes and case law (`LegalRetriever`, BGE-small).
5. Run `AirLLMClassifier` on `data/sample.csv` **with vs. without** the retriever.
6. Compare macro-F1 and inspect the `reasoning` field — with RAG it should cite USC sections, UK Acts, and/or case names.

**Backend:** [AirLLM](https://github.com/lyogavin/airllm) — loads one transformer layer at a time from disk → runs 7B/70B LLMs on tiny GPUs / Mac Silicon. MLX auto-selected on macOS. Generation is slow (seconds-to-minutes per row).

Prereqs:
- `uv sync && uv sync --extra airllm` (CUDA: `--extra airllm-cuda`; Mac: `--extra airllm-mlx`).
- Copy `.env.example` → `.env` and add `COURTLISTENER_API_TOKEN=...` if you have one.
- Open the notebook with the `crimellm` kernel.
- ~14 GB free disk for Qwen2.5-7B weights.

In [12]:
import os
from pathlib import Path

from crimellm import (
    AirLLMClassifier,
    LegalRetriever,
    fetch_us_code_sections,
    download_courtlistener,
    load_env,
    load_jsonl,
    load_dataset_from_csv,
)

# Load secrets from `.env` (CourtListener / govinfo / Anthropic tokens).
# Copy `.env.example` to `.env` at the project root and fill in values.
env_path = load_env()
print(f'.env loaded: {env_path}' if env_path else 'no .env found (anon access only)')
print(f"  COURTLISTENER_API_TOKEN: {'set' if os.environ.get('COURTLISTENER_API_TOKEN') else 'unset (anon, 5k/day)'}")
print(f"  GOVINFO_API_KEY       : {'set' if os.environ.get('GOVINFO_API_KEY') else 'unset (DEMO_KEY, 30/hr)'}")

CORPORA = Path('../data/corpora')
CORPORA.mkdir(parents=True, exist_ok=True)
USC_BASE = CORPORA / 'usc18_demo'
CL_BASE = CORPORA / 'cl_demo'
COMBINED_BASE = CORPORA / 'legal_combined'

.env loaded: /Users/paolobozzini/Desktop/CrimeLLM/.env
  COURTLISTENER_API_TOKEN: set
  GOVINFO_API_KEY       : unset (DEMO_KEY, 30/hr)


## 1. Ingest USC Title 18 — curated criminal-law sections

Hand-picked statutes covering the crime types in `sample.csv`. For a full ingest, see `download_us_code(...)` which walks all 1939 granules of Title 18 via api.govinfo.gov (needs a free [api.data.gov key](https://api.data.gov/signup/) — `DEMO_KEY` is rate-limited to 30 req/hour).

In [3]:
TITLE_18_CRIME_GRANULES = [
    'USCODE-2023-title18-partI-chap51-sec1111',   # § 1111. Murder
    'USCODE-2023-title18-partI-chap51-sec1112',   # § 1112. Manslaughter
    'USCODE-2023-title18-partI-chap103-sec2113',  # § 2113. Bank robbery
    'USCODE-2023-title18-partI-chap31-sec641',    # § 641. Public money/property theft
    'USCODE-2023-title18-partI-chap31-sec659',    # § 659. Interstate-shipment theft
    'USCODE-2023-title18-partI-chap47-sec1001',   # § 1001. False statements
    'USCODE-2023-title18-partI-chap63-sec1341',   # § 1341. Mail fraud
    'USCODE-2023-title18-partI-chap63-sec1343',   # § 1343. Wire fraud
    'USCODE-2023-title18-partI-chap25-sec471',    # § 471. Counterfeiting US obligations
    'USCODE-2023-title18-partI-chap7-sec113',     # § 113. Assault (maritime/territorial)
    'USCODE-2023-title18-partI-chap41-sec875',    # § 875. Interstate threats
]
if not (USC_BASE.parent / (USC_BASE.name + '.jsonl')).exists():
    n = fetch_us_code_sections(USC_BASE, TITLE_18_CRIME_GRANULES, year=2023, title='18')
    print(f'wrote {n} statute records')
else:
    print(f'already exists: {USC_BASE}.jsonl')

USCODE-2023-title18: 100%|██████████| 11/11 [00:02<00:00,  5.37it/s]

[corpora] wrote 11 records → ../data/corpora/usc18_demo.jsonl
wrote 11 statute records


## 2. Ingest CourtListener — real case-law judgments

`download_courtlistener` is a **two-hop** pipeline:

1. `/api/rest/v4/search/?type=o&q=<query>` — Elasticsearch over opinions, returns case name, court, citation, dateFiled, opinion id, and a highlighted snippet. Anon-friendly.
2. `/api/rest/v4/opinions/{id}/` — fetches the full `plain_text` body for each hit. **Requires a token** (set `COURTLISTENER_API_TOKEN` in `.env`).

If no token is set, falls back to snippet-only (still useful for retrieval). With token, builds out to 8 KB of body text per case. Includes pacing + 429 retry/backoff — burst limits are real.

Seed query targets the federal-crime domain so the cases overlap our classifier.

In [ ]:
if not (CL_BASE.parent / (CL_BASE.name + '.jsonl')).exists():
    n = download_courtlistener(
        CL_BASE,
        query='bank robbery OR mail fraud OR wire fraud OR burglary OR aggravated assault OR threats OR forgery',
        max_docs=25,
        order_by='score desc',
    )
    print(f'wrote {n} judgment records')
else:
    print(f'already exists: {CL_BASE}.jsonl')

In [ ]:
usc_records = load_jsonl(str(USC_BASE) + '.jsonl')
cl_records = load_jsonl(str(CL_BASE) + '.jsonl')
print(f'{len(usc_records)} statutes  +  {len(cl_records)} judgments  =  {len(usc_records)+len(cl_records)} total\n')
print('--- statutes ---')
for r in usc_records:
    print(f"  {r['citation']:>20}  {r['metadata']['heading'][:60]}")
print('\n--- cases (first 8) ---')
for r in cl_records[:8]:
    md = r['metadata']
    print(f"  {md['date_filed']:<12}  {md['court_id']:<12}  {md['case_name'][:60]}")

## 3. Ingest UK Acts of Parliament — legislation.gov.uk

`download_uk_legislation` pulls whole-act CLML XML from `legislation.gov.uk/{type}/{year}/{number}/data.xml` — Open Government Licence, no key, no rate limit, ~1 HTTP request per Act. Iterates `<P1>` body sections (skips schedules), extracts heading + statutory text.

Default `UK_CRIMINAL_ACTS` covers the main criminal-law statutes (Fraud Act 2006, Theft Act 1968, Misuse of Drugs Act 1971, Criminal Damage Act 1971, Offences against the Person Act 1861, Public Order Act 1986, Sexual Offences Act 2003, Modern Slavery Act 2015, Bribery Act 2010).

In [4]:
from crimellm import download_uk_legislation, UK_CRIMINAL_ACTS

UK_BASE = CORPORA / 'uk_demo'
if not (UK_BASE.parent / (UK_BASE.name + '.jsonl')).exists():
    n = download_uk_legislation(UK_BASE, statutes=UK_CRIMINAL_ACTS)
    print(f'wrote {n} UK statute sections')
else:
    print(f'already exists: {UK_BASE}.jsonl')

uk_records = load_jsonl(str(UK_BASE) + '.jsonl')
from collections import Counter
print('\nsections per Act:')
for act, c in Counter(r['metadata']['act_title'] for r in uk_records).items():
    print(f'  {c:>3}  {act}')

uk_legislation:  56%|█████▌    | 5/9 [00:00<00:00,  7.59it/s]

[corpora] skip ukpga/1861/100: 301


uk_legislation: 100%|██████████| 9/9 [00:12<00:00,  1.35s/it]

[corpora] wrote 256 records from 9 Acts → ../data/corpora/uk_demo.jsonl
wrote 256 UK statute sections

sections per Act:
   14  Fraud Act 2006
   35  Theft Act 1968
   30  Misuse of Drugs Act 1971
   11  Criminal Damage Act 1971
   29  Public Order Act 1986
  102  Sexual Offences Act 2003
   17  Modern Slavery Act 2015
   18  Bribery Act 2010


## 3. Build a combined FAISS index

One vector store over both sources — at retrieval time the LLM sees a mix of statutes and case law, ranked by cosine similarity. `BAAI/bge-small-en-v1.5` (384-dim) handles legal English well enough for small/medium corpora.

In [ ]:
all_records = usc_records + cl_records + uk_records
print(f'{len(usc_records)} USC + {len(cl_records)} CL + {len(uk_records)} UK = {len(all_records)} total')

if not (COMBINED_BASE.parent / (COMBINED_BASE.name + '.faiss')).exists():
    retriever = LegalRetriever.build(all_records, COMBINED_BASE,
                                     embedding_model='BAAI/bge-small-en-v1.5')
else:
    retriever = LegalRetriever.load(COMBINED_BASE)
print(f'index size: {len(retriever)}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

index size: 267


## 4. Retrieval sanity check — mixed sources

Criminal queries should pull the right USC section AND analogous cases; benign queries should score uniformly low.

In [10]:
for q in [
    'he broke into the shop at night and stole money from the till',
    'she defrauded investors with false statements about company finances',
    'he dishonestly appropriated property belonging to another',
    'they bribed a foreign official to win a contract',
    'the defendant assaulted a federal officer during the arrest',
    'she went for a walk in the park and read a book',
]:
    print(f'> {q}')
    for h in retriever.retrieve(q, k=4):
        label = h.metadata.get('heading') or h.metadata.get('case_name', '')
        print(f'  {h.score:.3f}  [{h.source:>14}]  {h.citation[:30]:<30}  {label[:55]}')
    print()

> he broke into the shop at night and stole money from the till
  0.559  [uk_legislation]  Theft Act 1968, s.25            Going equipped for stealing, etc.
  0.551  [uk_legislation]  Theft Act 1968, s.24            Scope of offences relating to stolen goods.
  0.543  [       us_code]  18 U.S.C. § 2113                §2113. Bank robbery and incidental crimes
  0.542  [uk_legislation]  Theft Act 1968, s.9             Burglary.

> she defrauded investors with false statements about company finances
  0.624  [uk_legislation]  Theft Act 1968, s.17            False accounting.
  0.621  [uk_legislation]  Theft Act 1968, s.19            False statements by company directors, etc.
  0.593  [uk_legislation]  Misuse of Drugs Act 1971, s.35  Financial provisions.
  0.587  [       us_code]  18 U.S.C. § 1341                §1341. Frauds and swindles

> he dishonestly appropriated property belonging to another
  0.797  [uk_legislation]  Theft Act 1968, s.2             “Dishonestly”
  0.712  [       

## 5. Classify with vs. without RAG — AirLLM backend

Same model, same system prompt, only the user message changes (retrieved context block prepended). First run downloads ~14 GB of Qwen2.5-7B-Instruct weights — be patient.

**Generation is slow.** AirLLM streams layers from disk → seconds per token. For a 6-row test set expect ~10-30 minutes on Mac MLX, faster on CUDA with `compression='4bit'`. Lower `max_new_tokens` (default 120) to trade detail for speed.

In [13]:
sample = Path("../data/sample.csv")
ds = load_dataset_from_csv(path=sample)
texts = list(ds['test']['text'])
labels = list(ds['test']['label'])
print(f'{len(texts)} test rows')

Casting the dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

4 test rows


In [14]:
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'  # or 'Qwen/Qwen2.5-3B-Instruct' (~6 GB) for faster smoke tests

baseline = AirLLMClassifier(model_id=MODEL_ID)
ragclf = AirLLMClassifier(model_id=MODEL_ID, retriever=retriever)

[crimellm] mlx backend does not support '4bit' compression; falling back to compression=None.


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

found_layers:{'model.embed_tokens.': True, 'model.layers.0.': True, 'model.layers.1.': True, 'model.layers.2.': True, 'model.layers.3.': True, 'model.layers.4.': True, 'model.layers.5.': True, 'model.layers.6.': True, 'model.layers.7.': True, 'model.layers.8.': True, 'model.layers.9.': True, 'model.layers.10.': True, 'model.layers.11.': True, 'model.layers.12.': True, 'model.layers.13.': True, 'model.layers.14.': True, 'model.layers.15.': True, 'model.layers.16.': True, 'model.layers.17.': True, 'model.layers.18.': True, 'model.layers.19.': True, 'model.layers.20.': True, 'model.layers.21.': True, 'model.layers.22.': True, 'model.layers.23.': True, 'model.layers.24.': True, 'model.layers.25.': True, 'model.layers.26.': True, 'model.layers.27.': True, 'model.layers.28.': True, 'model.layers.29.': True, 'model.layers.30.': True, 'model.layers.31.': True, 'model.norm.': True, 'lm_head.': True}
saved layers already found in /Users/paolobozzini/.cache/huggingface/hub/models--mistralai--Mist

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

found_layers:{'model.embed_tokens.': True, 'model.layers.0.': True, 'model.layers.1.': True, 'model.layers.2.': True, 'model.layers.3.': True, 'model.layers.4.': True, 'model.layers.5.': True, 'model.layers.6.': True, 'model.layers.7.': True, 'model.layers.8.': True, 'model.layers.9.': True, 'model.layers.10.': True, 'model.layers.11.': True, 'model.layers.12.': True, 'model.layers.13.': True, 'model.layers.14.': True, 'model.layers.15.': True, 'model.layers.16.': True, 'model.layers.17.': True, 'model.layers.18.': True, 'model.layers.19.': True, 'model.layers.20.': True, 'model.layers.21.': True, 'model.layers.22.': True, 'model.layers.23.': True, 'model.layers.24.': True, 'model.layers.25.': True, 'model.layers.26.': True, 'model.layers.27.': True, 'model.layers.28.': True, 'model.layers.29.': True, 'model.layers.30.': True, 'model.layers.31.': True, 'model.norm.': True, 'lm_head.': True}
saved layers already found in /Users/paolobozzini/.cache/huggingface/hub/models--mistralai--Mist

In [15]:
results_base = baseline.classify_many(texts)
results_rag = ragclf.classify_many(texts)

running layers: 100%|██████████| 32/32 [00:07<00:00,  4.06it/s]


In [17]:
import pandas as pd
from sklearn.metrics import f1_score

LABEL_NAMES = ['no', 'yes', 'unclear']
to_id = lambda s: LABEL_NAMES.index(s) if s in LABEL_NAMES else 2

pred_base = [to_id(r.label) for r in results_base]
pred_rag = [to_id(r.label) for r in results_rag]

print(f'macro-F1 baseline: {f1_score(labels, pred_base, average="macro", zero_division=0):.3f}')
print(f'macro-F1 + RAG  : {f1_score(labels, pred_rag, average="macro", zero_division=0):.3f}')
pd.set_option('display.max_colwidth', None)   # don't truncate cell contents
pd.set_option('display.max_rows', None)       # show all rows
pd.set_option('display.width', None)
df = pd.DataFrame({
    'text': texts,
    'gold': [LABEL_NAMES[i] for i in labels],
    'base_label': [r.label for r in results_base],
    'base_reason': [r.reasoning for r in results_base],
    'rag_label': [r.label for r in results_rag],
    'rag_reason': [r.reasoning for r in results_rag],
})
display(df)

macro-F1 baseline: 1.000
macro-F1 + RAG  : 1.000


,text,gold,base_label,base_reason,rag_label,rag_reason
0,Someone was in the room but it's not clear what they did.,unclear,unclear,The text does not specify an unlawful act.,unclear,The text does not describe a specific action or behavior.
1,We donated our old clothes to the shelter.,no,no,The text describes a lawful act of donating old clothes to a shelter.,no,The text describes a lawful act of donating old clothes.
2,I forged my manager's signature on the expense form.,yes,yes,"The text describes the act of forging a signature, which is a crime.",yes,"The text describes the act of forging a signature, which is a crime under typical legal standards."
3,He broke into the shop and emptied the till.,yes,yes,"The text describes a clear act of burglary, including breaking into a shop and stealing money from the till.",yes,"The text describes a clear unlawful act of burglary, as defined in Theft Act 1968, s.9."
